# Retinal Disease Detection Using Deep Learning on OCT Images
## Part 2: Custom CNN, Transfer Learning & Explainable AI (Grad-CAM)
**Domain:** Healthcare & Life Sciences – Medical Imaging (Ophthalmology)


In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

from src.config import config
from src.dataset import build_dataloaders
from src.models.model_factory import build_model, count_parameters
from src.engine.trainer import ModelTrainer
from src.engine.evaluator import ModelEvaluator
from src.explainability.gradcam import GradCAM

print(f'Active Device: {config.device}')


---
### 1. Architectural Parameter Comparison
Evaluating parameters and complexity across Custom CNN and Transfer Learning backbones.


In [ ]:
architectures = ['custom_cnn', 'resnet18', 'vgg16', 'mobilenetv2', 'densenet121']
table = []
for arch in architectures:
    m, _ = build_model(arch, num_classes=4, pretrained=False)
    p = count_parameters(m)
    table.append({
        'Architecture': arch,
        'Total Parameters (M)': round(p['total_parameters'] / 1e6, 2),
        'Trainable Parameters (M)': round(p['trainable_parameters'] / 1e6, 2)
    })
df_params = pd.DataFrame(table)
df_params


---
### 2. Explainable AI with Grad-CAM
Generating class activation maps on a sample OCT image to verify pathological focus.


In [ ]:
# Load ResNet-18 model and Grad-CAM extractor
model, target_layer = build_model('resnet18', num_classes=4, pretrained=True)
gradcam = GradCAM(model, target_layer)

# Load a sample CNV image
sample_path = list((config.samples_dir / 'CNV').glob('*.jpeg'))[0]
raw_img = Image.open(sample_path).convert('RGB')

tf = config.mean  # verify normalization
from torchvision import transforms
input_tensor = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=config.mean, std=config.std)
])(raw_img).unsqueeze(0)

cam, pred_class, conf = gradcam.generate_cam(input_tensor)
overlay = GradCAM.overlay_heatmap(raw_img, cam, alpha=0.5)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(raw_img)
axes[0].set_title('Original OCT Scan', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(cam, cmap='jet')
axes[1].set_title('Grad-CAM Heatmap', fontsize=12, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(overlay)
axes[2].set_title(f'Diagnostic Overlay\nPrediction: {config.classes[pred_class]} ({conf*100:.1f}%)', fontsize=12, fontweight='bold')
axes[2].axis('off')
plt.tight_layout()
plt.show()
